# GT-PAK 350Y (Wang) (PLAN step 5)
Data: Wang (2015) Table C.1, SRP column (0.427 m, air/water). Model code: `../packlib.py` (D47).
Decisions used: D49 and D53 (fluid properties), D55 (dry rows), D56 (step 5 rules, Level 0 constants, void fraction).

In [ ]:
# 1.1 load the library and the data

import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import packlib

df = pd.read_csv('../srp_data/wang2015_tableC1.csv')
data = df[df['packing'] == 'GTC350Y'].copy()
data['holdup'] = data['holdup_frac']
print(data.shape)
print(data['run_header'].value_counts())

In [ ]:
# 1.2 water properties per row from each row's liquid temperature (D49, D56)

water = pd.read_csv('../srp_data/properties/nist_water_0_40C.csv')
data['rho_L'] = np.interp(data['Tliq_in_C'], water['T_C'], water['rho_kg_m3'])
data['mu_L'] = np.interp(data['Tliq_in_C'], water['T_C'], water['mu_Pa_s'])
print(data['Tliq_in_C'].min(), data['Tliq_in_C'].max())

In [ ]:
# 1.3 air properties per row at the mean of air in and air out (D53, D56)
# density: ideal gas, dry air, 101.325 kPa assumed. viscosity: sutherland, white's constants

P = 101325
M_air = 0.0289644
R = 8.314462

T_gas = (data['Tair_in_C'] + data['Tair_out_C']) / 2 + 273.15
data['rho_G'] = P * M_air / (R * T_gas)
data['mu_G'] = 1.716e-5 * (T_gas / 273.15) ** 1.5 * (273.15 + 110.4) / (T_gas + 110.4)
print(data[['rho_G', 'mu_G']].describe().round(7))

In [ ]:
# 1.4 standard columns, dry rows (D55) and bad hold-ups (D56 ii)

data['system'] = 'air/water'
data['F_Pa05'] = data['FG_Pa05']
data['uG_ms'] = data['F_Pa05'] / np.sqrt(data['rho_G'])
data['dpdz_Pa_m'] = data['dPdz_Pa_m']
data['uL_m3_m2h'] = data['L_m3_m2h']

dry = data[data['L_m3_m2h'] == 0].copy()
# rows with no temperatures have no gas properties, so they can't be used (decision pending, D57)
print('dry rows without temperatures:', dry['rho_G'].isna().sum())
dry = dry[dry['rho_G'].notna()]
dry_fit = dry.drop(dry['F_Pa05'].idxmin())
wet = data[data['L_m3_m2h'] > 0].copy()
wet.loc[~(wet['holdup'] > 0), 'holdup'] = np.nan

print(dry.shape, dry_fit.shape, wet.shape, 'missing hold-up', wet['holdup'].isna().sum())

In [ ]:
# 1.5 curves (D56 i): within each run, loads less than 1.5 m3/m2h apart are one curve

labels = []
for i in wet.index:
    run = wet.loc[i, 'run_header']
    loads = np.sort(wet.loc[wet['run_header'] == run, 'L_m3_m2h'].unique())
    # start a new group wherever the gap to the previous load is more than 1.5
    group = np.cumsum(np.r_[0, np.diff(loads) > 1.5])
    g = group[np.searchsorted(loads, wet.loc[i, 'L_m3_m2h'])]
    labels.append(run + ' | ' + str(round(loads[group == g].mean(), 1)))
wet['curve_id'] = labels

curve = wet[['system', 'curve_id', 'uL_m3_m2h', 'F_Pa05', 'uG_ms', 'dpdz_Pa_m', 'holdup',
             'rho_L', 'mu_L', 'rho_G', 'mu_G']].copy()
curve = curve.sort_values(['curve_id', 'F_Pa05']).reset_index(drop=True)
print(curve.groupby('curve_id').size())

In [ ]:
# 2.1 settings. a: wang table 5.1. eps: provisional: tsai eq 6.2, 1 - a x 0.1 mm / 2 (hypothesis, D56)
# level 0 constants: borrowed: billet 1995 table 4.7, mellapak 250Y (D56)
# reference water for billet's flood hold-up: nist at 20 C (D53)

C0 = {'Cp': 0.292, 'Clp': 3.157, 'Cfp': 2.464, 'Ch': 0.554}
borrowed = True

s = packlib.make_settings(ap=350, eps=0.98, dc=0.427,
                          rho_G=curve['rho_G'].mean(), mu_G=curve['mu_G'].mean(),
                          props={'air/water': (curve['rho_L'].mean(), curve['mu_L'].mean())},
                          C0=C0, dry=dry_fit, rho_W=998.21, mu_W=1.0014e-3)
print(s['ap'], s['eps'], s['C0'], 'borrowed:', borrowed)

In [ ]:
# 2.2 level 0 on every row, and on the dry rows

pred0 = packlib.billet_rows(curve, C0, s)
dp_err = 100 * (pred0['dp'] - curve['dpdz_Pa_m']) / curve['dpdz_Pa_m']
hold_err = 100 * (pred0['hold'] - curve['holdup']) / curve['holdup']
print('answered', pred0['dp'].notna().sum(), 'of', len(curve))
print('dp abs / bias %', round(dp_err.abs().mean(), 1), round(dp_err.mean(), 1))
print('hold abs / bias %', round(hold_err.abs().mean(), 1), round(hold_err.mean(), 1))

s_dry = dict(s)
s_dry['rho_G'] = dry['rho_G'].values
s_dry['mu_G'] = dry['mu_G'].values
dry_err = 100 * (packlib.dry_dp(dry['uG_ms'].values, C0, s_dry) - dry['dpdz_Pa_m'].values) / dry['dpdz_Pa_m'].values
print('dry dp abs / bias %', round(np.abs(dry_err).mean(), 1), round(dry_err.mean(), 1))

In [ ]:
# 2.3 level 1 fitted on every row, to see the constants

lib = packlib.fit_constants(curve, ['hold'], s)
print({k: round(v, 4) for k, v in lib['air/water'].items()})

In [ ]:
# 3.1 the tests. water only, so no liquid-transfer test

splits = packlib.make_splits(curve)
for name, pairs in splits.items():
    print(name, '|', len(pairs), 'folds |', sum(len(test) for train, test in pairs), 'test rows')

In [ ]:
# 3.2 level 0, level 1 and the two fixed level 2 forms on every test

tests = packlib.run_tests(splits, s, ['L0', 'L1', 'L2 linear', 'L2 network'])

In [ ]:
# 3.3 the auto-tuned level 2 (D46) on leave one curve out only

tuned = packlib.run_tests({'leave one curve out': splits['leave one curve out']}, s, ['L2 tuned'])
tests['leave one curve out']['L2 tuned'] = tuned['leave one curve out']['L2 tuned']

In [ ]:
# 3.4 one table, errors on the rows every model answers
# borrowed level 0 constants are starred: their error may be inflated (D56)

results = packlib.results_table(tests, curve)
if borrowed:
    results['model'] = results['model'].replace('L0', 'L0*')
    print('* level 0 uses constants borrowed from another packing, so its error may be inflated')
results.round(1)

In [ ]:
# 3.5 spread (D34): level 1 against level 0, and level 2 against level 1, by curve. negative = better

print(packlib.spread_table(tests, curve, 'L0', ['L1']).round(1))
packlib.spread_table(tests, curve, 'L1', ['L2 linear', 'L2 network', 'L2 tuned']).round(1)